# Notebook 02 -- Ablation Study: 6-Configuration Evaluation on 225-Question Gold Set

Runs all six RAG pipeline configurations on the verified 225-question gold test
set and computes Token F1 + ROUGE-L for each. Results are persisted to Google
Drive so downstream analysis (Notebook 03) can pick them up without re-running
generation.

| Config | Embedding | LLM | Reranker | Description |
|--------|-----------|-----|---------|-------------|
| C1 | Base E5 | Base Qwen | No | Baseline |
| C2 | FT E5 | Base Qwen | No | Fine-tuned embeddings |
| C3 | FT E5 | Base Qwen | Yes | + Reranker |
| C4 | Base E5 | QLoRA Qwen | No | + QLoRA LLM |
| C5 | FT E5 | QLoRA Qwen | No | FT embed + QLoRA LLM |
| C6 | FT E5 | QLoRA Qwen | Yes | Full pipeline |

**Runtime:** Expect ~20-30 min per config on T4 (225 questions x ~5s/question).
Each config is checkpointed independently, so a Colab crash resumes from the
last completed config.

In [ ]:
# ====================================================================
# 1. SETUP -- installs, imports, seeds, constants, helper functions
# ====================================================================

# ---- Drive mount & secrets ----
from google.colab import drive, userdata
drive.mount("/content/drive")

try:
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    GITHUB_TOKEN = ""

# ---- Installs ----
!pip install -q transformers accelerate bitsandbytes peft sentence-transformers \
    faiss-cpu rank_bm25 rouge-score tqdm pyarrow 2>/dev/null

# ---- Imports ----
import gc
import json
import os
import pickle
import re
import sys
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq
import torch
from tqdm.auto import tqdm

# ---- Seeds ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ---- Paths ----
DRIVE_ROOT   = Path("/content/drive/MyDrive/hukuk-rag")
BM25_PATH    = DRIVE_ROOT / "indexes" / "bm25.pkl"
BASE_FAISS   = DRIVE_ROOT / "indexes" / "faiss.index"
BASE_MAP     = DRIVE_ROOT / "indexes" / "faiss.mapping.pkl"
FT_FAISS     = DRIVE_ROOT / "indexes" / "finetuned" / "faiss_ft.index"
FT_MAP       = DRIVE_ROOT / "indexes" / "finetuned" / "faiss_ft.mapping.pkl"
CHUNKS_PATH  = DRIVE_ROOT / "data" / "processed" / "chunks_filtered.parquet"
FT_E5_PATH   = DRIVE_ROOT / "models" / "e5-checkpoints" / "checkpoint-10000"
QLORA_PATH   = DRIVE_ROOT / "models" / "qwen-qlora-v2" / "checkpoints" / "checkpoint-322"
RERANKER_PATH = DRIVE_ROOT / "models" / "reranker-finetuned" / "model"
RESULTS_DIR  = DRIVE_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT = Path("/content/hukuk-rag")
GOLD_PATH    = PROJECT_ROOT / "data" / "gold" / "gold_test_set.json"

# ---- Constants ----
TURKISH_LOWER_MAP = str.maketrans("\u0130I\u00d6\u00dc\u00c7\u015e\u011e",
                                  "i\u0131\u00f6\u00fc\u00e7\u015f\u011f")

TURKISH_STOPWORDS = frozenset({
    "bir", "bu", "da", "de", "ve", "ile", "i\u00e7in", "olan", "olarak", "gibi",
    "daha", "en", "\u00e7ok", "her", "kadar", "sonra", "\u00f6nce", "ise", "ya",
    "ne", "nas\u0131l", "neden", "nerede", "kim", "hangi", "o", "\u015fu", "ben",
    "sen", "biz", "siz", "onlar", "mi", "mu", "m\u00fc", "m\u0131", "dir", "d\u0131r",
    "dur", "d\u00fcr", "tir", "t\u0131r", "tur", "t\u00fcr", "ki", "ama", "ancak",
    "fakat", "lakin", "veya", "yahut", "hem", "\u00fczere", "g\u00f6re", "kar\u015f\u0131",
    "aras\u0131nda", "taraf\u0131ndan", "dolay\u0131", "halde", "ra\u011fmen", "itibaren",
    "de\u011fil", "var", "yok", "eden", "eder", "etti", "oldu",
    "olur", "olmu\u015f", "iken", "olup", "buna", "\u015f\u00f6yle",
    "b\u00f6yle", "\u00f6yle", "ayn\u0131", "baz\u0131", "bir\u00e7ok", "di\u011fer", "ba\u015fka",
})

SYSTEM_PROMPT = (
    "Sen bir T\u00fcrk hukuku uzman\u0131s\u0131n. Soruyu verilen ba\u011flam paragraflar\u0131n\u0131 "
    "kullanarak k\u0131sa ve \u00f6z \u015fekilde yan\u0131tla (2-3 c\u00fcmle). \u0130lgili kanun "
    "maddelerine at\u0131fta bulun. Ba\u011flamda bilgi yoksa "
    "'Bu konuda yeterli bilgi bulunamad\u0131' de."
)

# ---- Helper functions (inlined for Colab portability) ----

_RE_PUNCT = re.compile(r'[^\w\s]')
_RE_MULTI_SPACE = re.compile(r'\s+')


def normalize_turkish(text):
    """Normalize Turkish text: locale-aware lowercase, remove punct, collapse whitespace."""
    text = text.translate(TURKISH_LOWER_MAP).lower()
    text = _RE_PUNCT.sub(' ', text)
    text = _RE_MULTI_SPACE.sub(' ', text).strip()
    return text


def turkish_tokenize(text):
    """Tokenize with Turkish lowercasing and stopword removal."""
    text = normalize_turkish(text)
    return [t for t in text.split() if t not in TURKISH_STOPWORDS and len(t) > 1]


def bm25_search(query, bm25_idx, bm25_map, k=50):
    """BM25 sparse search returning list of (chunk_id, score, text)."""
    tokens = turkish_tokenize(query)
    scores = bm25_idx.get_scores(tokens)
    top_indices = np.argsort(scores)[-k:][::-1]
    results = []
    for idx in top_indices:
        if scores[idx] <= 0:
            continue
        chunk = bm25_map[idx]
        results.append((chunk["chunk_id"], float(scores[idx]), chunk["text"]))
    return results


def dense_search(query, faiss_idx, mapping, embed_model, k=50, nprobe=16):
    """FAISS dense search. mapping can be list of dicts or list of chunk_id strings."""
    faiss_idx.nprobe = nprobe
    qvec = embed_model.encode(
        [f"query: {query}"], normalize_embeddings=True
    ).astype(np.float32)
    scores, indices = faiss_idx.search(qvec, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        m = mapping[idx]
        if isinstance(m, dict):
            results.append((m["chunk_id"], float(score), m["text"]))
        else:
            # FT mapping: list of chunk_id strings; need chunk_text_map for text
            results.append((m, float(score), chunk_text_map.get(m, "")))
    return results


def rrf_merge(list_a, list_b, k=60, top_k=10):
    """Reciprocal Rank Fusion on two result lists of (chunk_id, score, text)."""
    scores = {}
    texts = {}
    for rank, (cid, _, text) in enumerate(list_a):
        scores[cid] = scores.get(cid, 0) + 1.0 / (k + rank + 1)
        if cid not in texts:
            texts[cid] = text
    for rank, (cid, _, text) in enumerate(list_b):
        scores[cid] = scores.get(cid, 0) + 1.0 / (k + rank + 1)
        if cid not in texts:
            texts[cid] = text
    sorted_ids = sorted(scores, key=scores.get, reverse=True)[:top_k]
    return [(cid, scores[cid], texts[cid]) for cid in sorted_ids]


def format_context(results, top_k=10):
    """Format retrieval results into numbered context string."""
    parts = []
    for i, (cid, score, text) in enumerate(results[:top_k], 1):
        parts.append(f"[{i}] {text}")
    return "\n\n".join(parts)


def build_prompt(question, context_str, tokenizer_fn):
    """Build chat-formatted prompt for Qwen."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Ba\u011flam:\n{context_str}\n\n"
                f"Soru: {question}\n\n"
                "L\u00fctfen yukar\u0131daki ba\u011flam\u0131 kullanarak soruyu yan\u0131tla. "
                "Hangi kaynaklardan ([1], [2], ...) yararland\u0131\u011f\u0131n\u0131 belirt."
            ),
        },
    ]
    return tokenizer_fn.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


@torch.inference_mode()
def generate_answer(prompt, model, tokenizer_fn, max_new_tokens=256,
                    temperature=0.1, top_p=0.9, repetition_penalty=1.0):
    """Generate answer from LLM."""
    inputs = tokenizer_fn(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        do_sample=True,
        repetition_penalty=repetition_penalty,
        pad_token_id=tokenizer_fn.eos_token_id,
    )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer_fn.decode(generated, skip_special_tokens=True).strip()


def compute_metrics(predictions, references):
    """Compute Token F1 and ROUGE-L."""
    from rouge_score import rouge_scorer

    # Token F1 (SQuAD-style)
    f1_scores = []
    for pred, ref in zip(predictions, references):
        pred_tokens = Counter(normalize_turkish(pred).split())
        ref_tokens = Counter(normalize_turkish(ref).split())
        if not pred_tokens or not ref_tokens:
            f1_scores.append(0.0)
            continue
        common = sum((pred_tokens & ref_tokens).values())
        if common == 0:
            f1_scores.append(0.0)
            continue
        precision = common / sum(pred_tokens.values())
        recall = common / sum(ref_tokens.values())
        f1_scores.append(2 * precision * recall / (precision + recall))

    # ROUGE-L
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
    rouge_scores = [
        scorer.score(normalize_turkish(ref), normalize_turkish(pred))["rougeL"].fmeasure
        for pred, ref in zip(predictions, references)
    ]

    return {
        "token_f1": float(np.mean(f1_scores)),
        "rouge_l": float(np.mean(rouge_scores)),
        "per_question_f1": [float(x) for x in f1_scores],
        "per_question_rouge": [float(x) for x in rouge_scores],
    }


print("Setup complete.")

## Load Shared Components

Loads BM25 index, fine-tuned FAISS index + mapping, chunk texts (for FT
mapping lookups), fine-tuned E5 embedding model, base Qwen 4-bit LLM,
and the 225-question gold test set.

The base FAISS index is also loaded for configs that need it (C1, C4).
Both FAISS indexes stay in CPU RAM; only the active embedding model and
LLM occupy GPU VRAM.

In [ ]:
# ---- Clone repo for gold set access ----
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/berkay-aktas/hukuk-rag.git"
if not PROJECT_ROOT.exists():
    !git clone {REPO_URL} {PROJECT_ROOT}
else:
    !cd {PROJECT_ROOT} && git pull

# ---- Gold test set ----
with open(GOLD_PATH, "r", encoding="utf-8") as f:
    gold_raw = json.load(f)
gold_data = gold_raw["questions"]
print(f"Gold set: {len(gold_data)} questions")

# ---- BM25 ----
with open(BM25_PATH, "rb") as f:
    bm25_data = pickle.load(f)
bm25_idx = bm25_data["index"]
bm25_map = bm25_data["mapping"]
print(f"BM25: {len(bm25_map):,} chunks")

# ---- Chunk text map (for FT FAISS mapping lookups) ----
# FT FAISS mapping is a list of chunk_id strings; we need chunk_id -> text
print("Building chunk text map from parquet (streaming)...")
chunk_text_map = {}
pf = pq.ParquetFile(str(CHUNKS_PATH))
for batch in pf.iter_batches(batch_size=100_000, columns=["chunk_id", "text"]):
    df = batch.to_pandas()
    for cid, txt in zip(df["chunk_id"], df["text"]):
        chunk_text_map[cid] = txt
    del df
print(f"Chunk text map: {len(chunk_text_map):,} entries")

# Also build a list for index-based access (FT FAISS uses positional lookup)
chunk_text_list = list(chunk_text_map.values())

# ---- Base FAISS index ----
import faiss
base_faiss = faiss.read_index(str(BASE_FAISS))
with open(BASE_MAP, "rb") as f:
    base_faiss_map = pickle.load(f)  # list of dicts with 'chunk_id', 'text'
print(f"Base FAISS: {base_faiss.ntotal:,} vectors")

# ---- FT FAISS index ----
ft_faiss = faiss.read_index(str(FT_FAISS))
with open(FT_MAP, "rb") as f:
    ft_faiss_map = pickle.load(f)  # list of chunk_id strings
print(f"FT FAISS: {ft_faiss.ntotal:,} vectors")

# ---- FT E5 embedding model ----
from sentence_transformers import SentenceTransformer
ft_e5 = SentenceTransformer(str(FT_E5_PATH))
print(f"FT E5 loaded from {FT_E5_PATH.name}")

# ---- Base E5 embedding model (load to CPU to save VRAM) ----
base_e5 = SentenceTransformer("intfloat/multilingual-e5-large", device="cpu")
print("Base E5 loaded (CPU)")

# ---- Base Qwen 4-bit ----
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

LLM_NAME = "Qwen/Qwen2.5-7B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {LLM_NAME} in 4-bit...")
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
base_llm = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
base_llm.eval()
print(f"Base LLM loaded. GPU: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

print("\nAll shared components loaded.")

## Configuration Definitions

Each configuration is a function that takes a question string and returns
(answer, retrieved_results). The six configs form an ablation grid:

- **C1 (Baseline):** Base E5 + BM25 + RRF -> Base Qwen
- **C2 (FT Embeddings):** FT E5 + BM25 + RRF -> Base Qwen
- **C3 (FT Embed + Reranker):** FT E5 + BM25 + RRF + Reranker -> Base Qwen
- **C4 (QLoRA LLM):** Base E5 + BM25 + RRF -> QLoRA Qwen
- **C5 (FT Embed + QLoRA):** FT E5 + BM25 + RRF -> QLoRA Qwen
- **C6 (Full Pipeline):** FT E5 + BM25 + RRF + Reranker -> QLoRA Qwen

The reranker is loaded on-demand (CPU) for C3 and C6, then freed.
The QLoRA adapter is merged on-demand for C4, C5, C6.

In [ ]:
# ---- Reranker helper (loaded on demand) ----

def load_reranker():
    """Load fine-tuned reranker on CPU."""
    from transformers import AutoModelForSequenceClassification, AutoTokenizer as AT
    print("Loading reranker on CPU...")
    rr_tokenizer = AT.from_pretrained(str(RERANKER_PATH))
    rr_model = AutoModelForSequenceClassification.from_pretrained(
        str(RERANKER_PATH)
    ).to("cpu")
    rr_model.eval()
    return rr_model, rr_tokenizer


@torch.inference_mode()
def rerank(query, results, rr_model, rr_tokenizer, top_k=10):
    """Rerank results using cross-encoder. Input: [[query, passage], ...]."""
    if not results:
        return results
    pairs = [[query, text] for (_, _, text) in results[:50]]
    inputs = rr_tokenizer(
        pairs, padding=True, truncation=True, max_length=512, return_tensors="pt"
    ).to("cpu")
    scores = rr_model(**inputs).logits.squeeze(-1).cpu().numpy()
    ranked = sorted(zip(results[:50], scores), key=lambda x: x[1], reverse=True)
    return [(cid, float(s), text) for (cid, _, text), s in ranked[:top_k]]


# ---- QLoRA helper ----

def load_qlora_llm():
    """Load QLoRA adapter on top of base LLM."""
    from peft import PeftModel
    print(f"Loading QLoRA adapter from {QLORA_PATH.name}...")
    qlora_model = PeftModel.from_pretrained(base_llm, str(QLORA_PATH))
    qlora_model.gradient_checkpointing_disable()
    qlora_model.eval()
    print(f"QLoRA loaded. GPU: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
    return qlora_model


# ---- Config pipeline functions ----

def rag_c1(question):
    """C1: Base E5 + BM25 + RRF -> Base Qwen."""
    d = dense_search(question, base_faiss, base_faiss_map, base_e5, k=50)
    b = bm25_search(question, bm25_idx, bm25_map, k=50)
    merged = rrf_merge(d, b, k=60, top_k=10)
    ctx = format_context(merged)
    prompt = build_prompt(question, ctx, tokenizer)
    answer = generate_answer(prompt, base_llm, tokenizer)
    return answer, merged


def rag_c2(question):
    """C2: FT E5 + BM25 + RRF -> Base Qwen."""
    d = dense_search(question, ft_faiss, ft_faiss_map, ft_e5, k=50)
    b = bm25_search(question, bm25_idx, bm25_map, k=50)
    merged = rrf_merge(d, b, k=60, top_k=10)
    ctx = format_context(merged)
    prompt = build_prompt(question, ctx, tokenizer)
    answer = generate_answer(prompt, base_llm, tokenizer)
    return answer, merged


def rag_c3(question, rr_model=None, rr_tokenizer=None):
    """C3: FT E5 + BM25 + RRF + Reranker -> Base Qwen."""
    d = dense_search(question, ft_faiss, ft_faiss_map, ft_e5, k=50)
    b = bm25_search(question, bm25_idx, bm25_map, k=50)
    merged = rrf_merge(d, b, k=60, top_k=50)
    reranked = rerank(question, merged, rr_model, rr_tokenizer, top_k=10)
    ctx = format_context(reranked)
    prompt = build_prompt(question, ctx, tokenizer)
    answer = generate_answer(prompt, base_llm, tokenizer)
    return answer, reranked


def rag_c4(question, qlora_model=None):
    """C4: Base E5 + BM25 + RRF -> QLoRA Qwen."""
    d = dense_search(question, base_faiss, base_faiss_map, base_e5, k=50)
    b = bm25_search(question, bm25_idx, bm25_map, k=50)
    merged = rrf_merge(d, b, k=60, top_k=10)
    ctx = format_context(merged)
    prompt = build_prompt(question, ctx, tokenizer)
    answer = generate_answer(prompt, qlora_model, tokenizer, repetition_penalty=1.2)
    return answer, merged


def rag_c5(question, qlora_model=None):
    """C5: FT E5 + BM25 + RRF -> QLoRA Qwen."""
    d = dense_search(question, ft_faiss, ft_faiss_map, ft_e5, k=50)
    b = bm25_search(question, bm25_idx, bm25_map, k=50)
    merged = rrf_merge(d, b, k=60, top_k=10)
    ctx = format_context(merged)
    prompt = build_prompt(question, ctx, tokenizer)
    answer = generate_answer(prompt, qlora_model, tokenizer, repetition_penalty=1.2)
    return answer, merged


def rag_c6(question, qlora_model=None, rr_model=None, rr_tokenizer=None):
    """C6: FT E5 + BM25 + RRF + Reranker -> QLoRA Qwen."""
    d = dense_search(question, ft_faiss, ft_faiss_map, ft_e5, k=50)
    b = bm25_search(question, bm25_idx, bm25_map, k=50)
    merged = rrf_merge(d, b, k=60, top_k=50)
    reranked = rerank(question, merged, rr_model, rr_tokenizer, top_k=10)
    ctx = format_context(reranked)
    prompt = build_prompt(question, ctx, tokenizer)
    answer = generate_answer(prompt, qlora_model, tokenizer, repetition_penalty=1.2)
    return answer, reranked


print("All 6 configuration pipelines defined.")

## Run Evaluation

Iterates over the 6 configs and runs each on all 225 gold questions.
Per-config results (predictions, references, metrics) are saved to Drive
immediately after completion, so a Colab crash only loses the in-progress
config.

Configs that need the reranker (C3, C6) load it on demand; configs that
need QLoRA (C4, C5, C6) load the adapter on demand. Components are freed
between configs where possible.

In [ ]:
# Output file names for each config (match the naming convention used downstream)
CONFIG_FILES = {
    "C1": "config1_predictions_fixed.json",
    "C2": "config2_finetuned_embeddings_predictions.json",
    "C3": "config3_predictions_fixed.json",
    "C4": "config4_predictions.json",
    "C5": "config5_best_predictions.json",
    "C6": "config6_full_predictions.json",
}

CONFIG_DESCRIPTIONS = {
    "C1": "Base E5 + BM25 + RRF -> Base Qwen",
    "C2": "FT E5 + BM25 + RRF -> Base Qwen",
    "C3": "FT E5 + BM25 + RRF + Reranker -> Base Qwen",
    "C4": "Base E5 + BM25 + RRF -> QLoRA Qwen",
    "C5": "FT E5 + BM25 + RRF -> QLoRA Qwen",
    "C6": "FT E5 + BM25 + RRF + Reranker -> QLoRA Qwen",
}

all_results = {}

# Pre-load optional components as None; load on demand
qlora_model = None
rr_model = None
rr_tokenizer = None

for config_name in ["C1", "C2", "C3", "C4", "C5", "C6"]:
    out_path = RESULTS_DIR / CONFIG_FILES[config_name]

    # Skip if already completed
    if out_path.exists():
        print(f"\n[skip] {config_name} already at {out_path.name}")
        with open(out_path, "r", encoding="utf-8") as f:
            all_results[config_name] = json.load(f)
        continue

    print(f"\n{'='*60}")
    print(f"Running {config_name}: {CONFIG_DESCRIPTIONS[config_name]}")
    print(f"{'='*60}")

    # Load reranker if needed (C3, C6)
    if config_name in ("C3", "C6") and rr_model is None:
        rr_model, rr_tokenizer = load_reranker()

    # Load QLoRA if needed (C4, C5, C6)
    if config_name in ("C4", "C5", "C6") and qlora_model is None:
        qlora_model = load_qlora_llm()

    predictions = []
    references = []

    for item in tqdm(gold_data, desc=config_name):
        question = item["question"]
        gold_answer = item["gold_answer"]

        try:
            if config_name == "C1":
                answer, _ = rag_c1(question)
            elif config_name == "C2":
                answer, _ = rag_c2(question)
            elif config_name == "C3":
                answer, _ = rag_c3(question, rr_model, rr_tokenizer)
            elif config_name == "C4":
                answer, _ = rag_c4(question, qlora_model)
            elif config_name == "C5":
                answer, _ = rag_c5(question, qlora_model)
            elif config_name == "C6":
                answer, _ = rag_c6(question, qlora_model, rr_model, rr_tokenizer)
        except Exception as e:
            print(f"  [ERROR] {question[:50]}... -> {e}")
            answer = "Hata olustu."

        predictions.append(answer)
        references.append(gold_answer)

    # Compute metrics
    metrics = compute_metrics(predictions, references)
    print(f"  Token F1: {metrics['token_f1']:.4f}  |  ROUGE-L: {metrics['rouge_l']:.4f}")

    # Save
    result_data = {
        "config": config_name,
        "description": CONFIG_DESCRIPTIONS[config_name],
        "predictions": predictions,
        "references": references,
        "metrics": {
            "token_f1": metrics["token_f1"],
            "rouge_l": metrics["rouge_l"],
        },
    }
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(result_data, f, ensure_ascii=False, indent=2)
    print(f"  Saved -> {out_path.name}")
    all_results[config_name] = result_data

    # Free reranker after C3 if C6 not yet run
    if config_name == "C3" and "C6" not in all_results:
        # Keep reranker alive for C6
        pass

# ---- Cleanup optional components ----
if qlora_model is not None:
    del qlora_model
if rr_model is not None:
    del rr_model, rr_tokenizer
gc.collect()
torch.cuda.empty_cache()

print("\nAll 6 configs evaluated.")

## Results

Load all result files from Drive and print a comparison table.
This cell can be re-run independently after all configs have completed
across multiple Colab sessions.

In [ ]:
import json
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/hukuk-rag")
RESULTS_DIR = DRIVE_ROOT / "results"

CONFIG_FILES = {
    "C1": "config1_predictions_fixed.json",
    "C2": "config2_finetuned_embeddings_predictions.json",
    "C3": "config3_predictions_fixed.json",
    "C4": "config4_predictions.json",
    "C5": "config5_best_predictions.json",
    "C6": "config6_full_predictions.json",
}

CONFIG_LABELS = {
    "C1": "Baseline",
    "C2": "+ FT Embeddings",
    "C3": "+ FT Embed + Reranker",
    "C4": "+ QLoRA LLM",
    "C5": "+ FT Embed + QLoRA",
    "C6": "Full Pipeline",
}

print(f"{'Config':<6} {'Description':<28} {'Token F1':>10} {'ROUGE-L':>10}")
print("-" * 58)

for cfg in ["C1", "C2", "C3", "C4", "C5", "C6"]:
    path = RESULTS_DIR / CONFIG_FILES[cfg]
    if not path.exists():
        print(f"{cfg:<6} {'MISSING':<28} {'---':>10} {'---':>10}")
        continue
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    m = data.get("metrics", {})
    tf1 = m.get("token_f1", 0)
    rl = m.get("rouge_l", 0)
    print(f"{cfg:<6} {CONFIG_LABELS[cfg]:<28} {tf1:>10.4f} {rl:>10.4f}")

print("-" * 58)
print("\nResults loaded from Drive. See Notebook 03 for statistical analysis.")